# Week 3 — Machine learning through one problem
**ESE · AI for Business and FinTech · 5 October 2026**

One problem, done properly, teaches more than a tour of algorithms. Today: *can we predict anything about next week's BTC from what we know today?* We build features, train two models against a trivial baseline, and — most importantly — evaluate them in the only way that means anything for time series. Then we turn a prediction into a decision.

In [ ]:
!pip -q install yfinance

In [ ]:
# --- course helper: price loader with fallbacks (run this cell once per session) ---
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def load_prices(tickers, start="2022-01-01", end=None, cache_dir="data"):
    """Daily close prices, one column per ticker.
    1) try yfinance (live)  2) try a CSV snapshot in data/  3) synthetic random walk (pipeline test only)."""
    import os
    os.makedirs(cache_dir, exist_ok=True)
    key = "_".join(t.replace("^", "") for t in tickers)
    snap = os.path.join(cache_dir, f"prices_{key}.csv")
    try:
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})
        px = px.dropna(how="all")
        if len(px) < 50:
            raise RuntimeError("empty download")
        px.to_csv(snap)
        print(f"[live] {px.shape[0]} rows from yfinance; snapshot saved to {snap}")
        return px
    except Exception as e:
        print(f"[warn] yfinance failed ({type(e).__name__}); trying snapshot")
    if os.path.exists(snap):
        px = pd.read_csv(snap, index_col=0, parse_dates=True)
        print(f"[snapshot] {px.shape[0]} rows from {snap}")
        return px
    print("[SYNTHETIC] no network and no snapshot: generating a random walk. Numbers below are NOT real.")
    rng = np.random.default_rng(0)
    idx = pd.bdate_range(start, end or pd.Timestamp.today().normalize())
    px = pd.DataFrame({t: 100 * np.exp(np.cumsum(rng.normal(0.0004, 0.02 if "USD" in t else 0.011, len(idx))))
                       for t in tickers}, index=idx)
    return px

In [ ]:
px = load_prices(["BTC-USD", "SPY"], start="2018-01-01").ffill().dropna()
d = pd.DataFrame(index=px.index)
d["ret"] = px["BTC-USD"].pct_change()
d["spy_ret"] = px["SPY"].pct_change()
d = d.dropna()
print(d.shape); d.tail(3)

## Block A — Features, targets, baselines

**Feature** = something known *at the time of prediction*. **Target** = the thing we want to know, which happens *after*. The entire discipline of ML on time series is keeping the wall between them intact.

We define two targets for the same features, because they behave very differently:
- **Direction**: is the return over the next 5 trading days positive?
- **Volatility regime**: will realised volatility over the next 5 days be above its trailing median?

In [ ]:
H = 5   # horizon in trading days

# Features: everything uses only past information (note the .shift(1) is NOT needed because rolling windows end at t inclusive, and the target starts at t+1)
f = pd.DataFrame(index=d.index)
f["ret_1"]   = d["ret"]
f["ret_5"]   = d["ret"].rolling(5).sum()
f["ret_21"]  = d["ret"].rolling(21).sum()
f["vol_5"]   = d["ret"].rolling(5).std()
f["vol_21"]  = d["ret"].rolling(21).std()
f["vol_ratio"] = f["vol_5"] / f["vol_21"]
f["spy_ret_5"] = d["spy_ret"].rolling(5).sum()
f["dow"]     = d.index.dayofweek

# Targets: strictly in the future (t+1 … t+H)
fwd_ret = d["ret"][::-1].rolling(H).sum()[::-1].shift(-1)   # sum of returns t+1..t+H (reverse-rolling, then shift) — verified by hand below
fwd_vol = d["ret"][::-1].rolling(H).std()[::-1].shift(-1)
y_dir = (fwd_ret > 0).astype(int).rename("y_dir")
y_vol = (fwd_vol > f["vol_21"].expanding().median()).astype(int).rename("y_vol")   # vs trailing (expanding) median: no future info

data = f.join([y_dir, y_vol, fwd_ret.rename("fwd_ret")]).dropna()
print(data.shape)
data.tail(3)

**🔍 CHECK — verify the target by hand.** Pick a date, and confirm that `fwd_ret` on that date equals the sum of `ret` over the *following* 5 rows. If it includes the current day, the target leaks. Do this every single time you build a forward-looking target; it takes one minute and it is the most common bug in trading ML.

In [ ]:
i = 1000
t = data.index[i]
manual = d["ret"].loc[t:].iloc[1:H+1].sum()
print(t.date(), "| fwd_ret in table:", round(data.loc[t, "fwd_ret"], 6), "| recomputed by hand:", round(manual, 6))
assert abs(manual - data.loc[t, "fwd_ret"]) < 1e-12, "target leaks or is misaligned"
print("OK — target uses only future rows")

In [ ]:
# Base rates: what does 'doing nothing clever' score?
for col in ["y_dir", "y_vol"]:
    print(f"{col}: share of 1s = {data[col].mean():.3f}  -> majority-class accuracy = {max(data[col].mean(), 1-data[col].mean()):.3f}")

The **baseline** is the number a model must beat. For direction, "always predict up" already scores ~55% because BTC drifted up over the sample. A model with 56% accuracy has learned almost nothing. Any assistant that reports "our model achieves 58% accuracy" without the base rate next to it has reported nothing.

## Block B — Two models, two ways of splitting

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

features = ["ret_1", "ret_5", "ret_21", "vol_5", "vol_21", "vol_ratio", "spy_ret_5", "dow"]
X = data[features]

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}

def evaluate(model, X, y, splits):
    accs, aucs = [], []
    for tr, te in splits:
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        accs.append(accuracy_score(y.iloc[te], p > 0.5)); aucs.append(roc_auc_score(y.iloc[te], p))
    return np.mean(accs), np.mean(aucs)

### 🔍 CHECK — the assistant's split
Asked to "evaluate the model with cross-validation", the assistant used the sklearn default: a **random shuffle**. For time series this is a leak: the model trains on Wednesday and Friday and is tested on Thursday, with overlapping 21-day windows on both sides. Compare with a **time-ordered** split where the test set is always in the future of the training set.

In [ ]:
rng = np.random.default_rng(0)
n = len(X)
shuffled = [(tr, te) for tr, te in [train_test_split(np.arange(n), test_size=0.2, random_state=s) for s in range(5)]]
ordered  = list(TimeSeriesSplit(n_splits=5, test_size=250).split(X))

rows = []
for target in ["y_dir", "y_vol"]:
    y = data[target]
    for name, m in models.items():
        a_s, u_s = evaluate(m, X, y, shuffled)
        a_o, u_o = evaluate(m, X, y, ordered)
        rows.append([target, name, a_s, u_s, a_o, u_o])
res = pd.DataFrame(rows, columns=["target", "model", "acc (shuffled)", "auc (shuffled)", "acc (time-ordered)", "auc (time-ordered)"]).round(3)
res["base rate"] = res["target"].map(lambda c: round(max(data[c].mean(), 1 - data[c].mean()), 3))
res

Read the table slowly. Three things should be visible:

1. **Shuffled scores are higher than time-ordered ones.** The difference is leakage, not skill.
2. **Direction is close to the base rate under an honest split.** With these features, 5-day direction of BTC is essentially unpredictable — which is what finance theory predicts and what an assistant will rarely tell you.
3. **Volatility regime is predictable** (AUC well above 0.5): volatility clusters. Same data, same models — the *question* decides whether ML has anything to offer. Choosing the question is the manager's job.

## Block C — Leakage on purpose

To recognise leakage you must see it once. We add a feature that is *almost* legitimate: a 5-day rolling volatility computed with a **centred** window (`center=True`), which is what an assistant may write when asked to "smooth" a series. Centred windows use future rows.

In [ ]:
X_leak = X.copy()
X_leak["vol_5_centred"] = d["ret"].rolling(5, center=True).std().reindex(X.index)
X_leak = X_leak.dropna(); y = data.loc[X_leak.index, "y_vol"]
ordered_l = list(TimeSeriesSplit(n_splits=5, test_size=250).split(X_leak))
a, u = evaluate(models["gradient boosting"], X_leak, y, ordered_l)
print(f"with the 'smoothed' feature — time-ordered acc {a:.3f}, auc {u:.3f}   (compare with the honest row above)")

**🔍 CHECK.** The score jumped with a time-ordered split — so the split did not protect you. Leakage through *features* is invisible to any validation scheme. The only defence is reading every feature and asking: *at time t, could I have computed this?* Write down, for each of the eight honest features, the latest row it uses.

## Block D — From prediction to decision

A probability is not a decision. To act you need a **decision rule** (a threshold) and the **cost of each error**. Example: a treasury desk that wants to hedge BTC exposure when a high-volatility week is coming. Hedging costs money (`c_hedge`); an unhedged high-vol week costs more (`c_miss`). The best threshold depends on that ratio, not on accuracy.

In [ ]:
# Walk-forward probabilities for the volatility target (out-of-sample only)
y = data["y_vol"]; m = models["gradient boosting"]
oos = pd.Series(index=X.index, dtype=float)
for tr, te in TimeSeriesSplit(n_splits=8, test_size=200).split(X):
    m.fit(X.iloc[tr], y.iloc[tr]); oos.iloc[te] = m.predict_proba(X.iloc[te])[:, 1]
oos = oos.dropna(); y_oos = y.loc[oos.index]

c_hedge, c_miss = 1.0, 4.0     # relative costs: hedge costs 1, missing a high-vol week costs 4
rows = []
for thr in np.arange(0.2, 0.81, 0.05):
    hedge = oos > thr
    cost = (hedge * c_hedge + (~hedge & (y_oos == 1)) * c_miss).sum()
    rows.append([thr, hedge.mean(), recall_score(y_oos, hedge), precision_score(y_oos, hedge, zero_division=0), cost])
dec = pd.DataFrame(rows, columns=["threshold", "share hedged", "recall (high-vol caught)", "precision", "total cost"]).round(3)
print("cost of never hedging:", (y_oos == 1).sum() * c_miss, "| cost of always hedging:", len(y_oos) * c_hedge)
dec

**🔍 CHECK.** Which threshold minimises cost? Does it change if `c_miss` is 2 instead of 4? Now the important question: *what KPI would you report to management for this system in production?* Accuracy is wrong (it does not price errors). Candidates: cost per week vs the always-hedge policy; recall at the chosen threshold; calibration (when the model says 70%, does it happen 70% of the time?). Pick one and defend it.

In [ ]:
# Calibration: does p=0.7 mean 70%?
bins = pd.cut(oos, [0, .3, .4, .5, .6, .7, 1.0])
cal = pd.DataFrame({"predicted": oos.groupby(bins).mean(), "observed": y_oos.groupby(bins).mean(), "n": oos.groupby(bins).size()}).round(3)
cal

**Homework brief** → see `week-03/homework.md`.

**Vocabulary you now own:** feature, target, horizon, base rate, baseline, train/test, time-ordered split, walk-forward, leakage (through split and through features), accuracy vs AUC vs precision/recall, threshold, calibration, cost-weighted decision.